# 要約が原文と食い違っていないか検査する

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/25_summarization/summary_faithfulness.ipynb)

NLP クックブック 25-07 ／ 要約と平易化

生成した要約に、原文にないことや、原文と矛盾することが書かれていないかを自動で確かめたい。

In [1]:
import re
import unicodedata
import pandas as pd
import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer

source = ("市は 14 日、来年 4 月から市内の全小中学校で給食費を無償化すると発表した。対象は約 1 万 2 千人で、年間の費用は約 6 億円を見込む。"
          "財源には、ふるさと納税の寄付金と、公共施設の統廃合で生じる削減分を充てる。"
          "市長は記者会見で「子育て世帯の負担を減らし、若い世代に選ばれるまちにしたい」と述べた。"
          "一方、市議会の一部からは「恒久的な財源の裏付けが弱い」との指摘が出ている。")
summaries = {
    "忠実": "市は来年 4 月から小中学校の給食費を無償化する。費用は年間約 6 億円を見込む。市議会の一部からは財源を不安視する声がある。",
    "矛盾あり": "市は来年 4 月から小中学校の給食費を無償化する。市議会は全会一致でこれを歓迎している。",
    "付け足しあり": "市は来年 4 月から小中学校の給食費を無償化する。隣の市でも同様の制度が始まっている。",
    "数値の誤り": "市は来年 4 月から小中学校の給食費を無償化する。費用は年間約 60 億円で、対象は約 1 万 2 千人である。",
}

nli_name = "MoritzLaurer/mDeBERTa-v3-base-mnli-xnli"
tok = AutoTokenizer.from_pretrained(nli_name)
nli = AutoModelForSequenceClassification.from_pretrained(nli_name).eval()

def nli_probs(premise, hypotheses):
    with torch.no_grad():
        return nli(**tok([premise] * len(hypotheses), hypotheses, return_tensors="pt", padding=True, truncation=True)).logits.softmax(-1)

rows = []
for label, summary in summaries.items():
    sents = [s for s in re.split(r"(?<=。)", summary) if s.strip()]
    for sent, p in zip(sents, nli_probs(source, sents)):
        verdict = "含意" if p[0] > 0.5 else "矛盾" if p[2] > 0.5 else "中立 (原文にない)"
        rows.append({"要約": label, "文": sent, "含意": round(p[0].item(), 2), "矛盾": round(p[2].item(), 2), "判定": verdict})
pd.set_option("display.max_colwidth", 60)
pd.DataFrame(rows)

,要約,文,含意,矛盾,判定
0,忠実,市は来年 4 月から小中学校の給食費を無償化する。,1.00,0.00,含意
1,忠実,費用は年間約 6 億円を見込む。,0.96,0.00,含意
2,忠実,市議会の一部からは財源を不安視する声がある。,1.00,0.00,含意
3,矛盾あり,市は来年 4 月から小中学校の給食費を無償化する。,1.00,0.00,含意
4,矛盾あり,市議会は全会一致でこれを歓迎している。,0.00,0.99,矛盾
5,付け足しあり,市は来年 4 月から小中学校の給食費を無償化する。,1.00,0.00,含意
6,付け足しあり,隣の市でも同様の制度が始まっている。,0.00,0.00,中立 (原文にない)
7,数値の誤り,市は来年 4 月から小中学校の給食費を無償化する。,1.00,0.00,含意
8,数値の誤り,費用は年間約 60 億円で、対象は約 1 万 2 千人である。,0.04,0.96,矛盾


In [2]:
# 数値と固有名詞は NLI が見逃しやすいので、原文に同じ表記があるかを別に確かめる
def numbers(text):
    text = unicodedata.normalize("NFKC", text)
    return set(re.findall(r"\d+(?:\.\d+)?\s*(?:万|億|千)?\s*(?:円|人|日|月|年|例目|%)?", text.replace(" ", "")))

src_nums = numbers(source)
for label, summary in summaries.items():
    missing = [n for n in numbers(summary) if n not in src_nums]
    print(f"{label}: 原文にない数値表現 = {missing}")

忠実: 原文にない数値表現 = []
矛盾あり: 原文にない数値表現 = []
付け足しあり: 原文にない数値表現 = []
数値の誤り: 原文にない数値表現 = ['60億円']


- 要約の各文を仮説、原文を前提として含意関係を判定する。含意なら裏付けあり、矛盾なら誤り、中立なら原文にない付け足し。
- NLI は桁違いの数値 (6 億と 60 億) のような誤りを見逃すことがある。数値・日付・固有名詞は、文字列として原文に存在するかを別に検査する。
- 原文が長いと NLI モデルの入力長を超える。原文を段落に分け、要約文ごとに最も含意の高い段落を採用する。
- 忠実であることと、良い要約であることは別。重要な内容を落としていないか (網羅性) は、参照要約との比較や人手で確かめる。